# 03 — Train Autoencoder
Unsupervised anomaly detection: train on normal traffic only.

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import torch

from src.config import (
    DEVICE, AE_LATENT_DIM, AE_HIDDEN_DIMS, AE_LR, AE_EPOCHS, AE_PATIENCE,
    AUTOENCODER_CHECKPOINT, PROCESSED_DIR, BATCH_SIZE
)
from src.models import Autoencoder
from src.dataset import make_loader
from src.train_utils import train_autoencoder
from src.visualize import plot_loss_curves

print(f'Device: {DEVICE}')

## Load preprocessed data

In [ ]:
with open(PROCESSED_DIR / 'data.pkl', 'rb') as f:
    data = pickle.load(f)

# Train autoencoder only on NORMAL traffic (unsupervised)
X_train_normal = data['X_train'][data['normal_mask_train']]
X_val = data['X_val']  # validate on all (to see reconstruction error on attacks too)
print(f'Normal training samples: {len(X_train_normal):,}')
print(f'Validation samples     : {len(X_val):,}')
input_dim = X_train_normal.shape[1]
print(f'Input dimension        : {input_dim}')

## Build model & loaders

In [ ]:
model = Autoencoder(input_dim=input_dim, hidden_dims=AE_HIDDEN_DIMS, latent_dim=AE_LATENT_DIM)
model = model.to(DEVICE)
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=AE_LR)

train_loader = make_loader(X_train_normal, shuffle=True, batch_size=BATCH_SIZE)
val_loader   = make_loader(X_val, data['y_bin_val'], shuffle=False, batch_size=BATCH_SIZE)

## Train

In [ ]:
history = train_autoencoder(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=DEVICE,
    epochs=AE_EPOCHS,
    patience=AE_PATIENCE,
    checkpoint_path=AUTOENCODER_CHECKPOINT,
)
print(f'\nBest checkpoint: {AUTOENCODER_CHECKPOINT}')

## Loss curves

In [ ]:
plot_loss_curves(history, title='Autoencoder Loss Curves', filename='ae_loss_curves.png')